# krig - Intro Notebook

Declarative device SDK: describe properties, compose pipelines, time-travel through state.

For a local checkout, run `./gradlew publishToMavenLocal` first, then load the sibling descriptor:
`%use @file[krig.json]`.
After krig is published as a Kotlin Notebook library, this cell can become `%use krig`.

In [ ]:
%use @file[krig.json]
// Covers: Device, DeviceManifest, descriptors, catalog, Timestamped,
//         ObservedValue, OperationOutcome, Timeline, Meta, Magix, simulation.

## 1. Inline device - read / write / actions

In [ ]:
object NotebookThermoContract : DeviceContractBuilder() {
    val temperature by mutableDoubleProperty()
    val setpoint by mutableDoubleProperty()
    val reset by action(MetaConverter.meta, MetaConverter.meta)
}

val notebookContext = krigNotebookContext("notebook-demo") {
    plugin(DeviceCatalog)
}
val thermoManifest = manifestOf("thermo", NotebookThermoContract)
notebookContext.registerManifests(listOf(thermoManifest))

var temperature = 22.0
var setpoint = 20.0
val thermoBackend = deviceBackend {
    reader(NotebookThermoContract.temperature) { temperature }
    writer(NotebookThermoContract.temperature) { value -> temperature = value }
    reader(NotebookThermoContract.setpoint) { setpoint }
    writer(NotebookThermoContract.setpoint) { value -> setpoint = value }
    action(NotebookThermoContract.reset) {
        temperature = 22.0
        null
    }
}

val thermo = runBlocking {
    device("thermo", thermoBackend, notebookContext) {
        manifest(thermoManifest)
    }
}
thermo

In [ ]:
// Manifest/catalog are the contract discovery surface
notebookContext.findManifest("thermo".asName())

In [ ]:
// OperationOutcome - faults as values, no try/catch
val outcome = runBlocking { thermo.readPropertyOutcome("temperature".asName()) }
outcome

In [ ]:
// Typed facade over the same contract
runBlocking {
    thermo.writeOutcome(NotebookThermoContract.temperature, 24.0)
    thermo.readOutcome(NotebookThermoContract.temperature)
}

In [ ]:
// Write through the Meta/control-plane boundary, re-read
runBlocking {
    thermo.writeProperty("temperature".asName(), metaOf(25.0))
    thermo.readProperty("temperature".asName())
}

## 2. Property descriptor

In [ ]:
// The manifest renderer shows contract shape; individual descriptors render too.
thermoManifest.properties.getValue("temperature".asName())

## 3. Device hub - attach, detach, reconcile

In [ ]:
val hubCtx = krigNotebookContext("hub-demo")
val hub = MutableDeviceHub("hub".asName(), hubCtx)

runBlocking {
    val child = device("child", hubCtx) {
        mutableProperty("ready", initial = true)
    }
    hub.attach("child".asName(), child)
}

println("Children: ${hub.children.keys}")
hub

## 4. Timeline - mergeable event streams

In [ ]:
// Timeline wraps a Flow<DeviceMessage> with merge combinators for multi-device views
val tl = thermo.timeline()
tl

In [ ]:
// Event log: cold, replayable source for time-travel and counterfactuals
val log = thermo.replayLog()
log

## 5. Simulation - virtual time

In [ ]:
// Simulation symbols are loaded by krig.json through krig-jupyter.
// Available here: DeterministicScheduler, ProcessDsl, Resource, Signal, SimulationSession.

In [ ]:
import kotlinx.coroutines.CoroutineScope
import kotlinx.coroutines.cancel
import kotlin.time.Duration.Companion.seconds

val sched = DeterministicScheduler()  // initialTimeMs defaults to 0
val scope = CoroutineScope(sched.asDispatcher())

// Process DSL: hold, waitUntil, request - coroutine-native, no yield sentinels
scope.process("ramp") {
    hold(1.seconds)
    hold(2.seconds)
}

// Advance virtual time 5 seconds - coroutines scheduled in that window all run
runBlocking { sched.advanceBy(5.seconds) }
sched.currentTimeMs

## 6. In-memory storage

In [ ]:
val storage = InMemoryEventJournal()

// Storage-backed PropertyHistory - replays recorded PropertyChangedMessages
val storageHistory = storage.propertyHistory(
    "thermo".asName(),
    "temperature".asName(),
    MetaConverter.double,
)
storageHistory

In [ ]:
// The history is lazy - it reads from storage on each flow subscription
runBlocking {
    storage.write(
        PropertyChangedMessage(
            time = Clock.System.now(),
            property = "temperature".asName(),
            value = metaOf(42.0),
            sourceDevice = "thermo".asName(),
            targetDevice = null,
        )
    )
    storageHistory.flowHistory().toList()
}

## 7. Cleanup

In [ ]:
scope.cancel()
hub.close()
hubCtx.close()
thermo.close()
notebookContext.close()